# Finite-sample oracle and controlled heterogeneity
**Task-aware amortized importance sampling — follow-up experiment (Stage 0)**

The earlier 4D/8D Gaussian-mixture experiments found that simple context-noise and proposal-width rules outperform the learned asymptotic allocator. This notebook tests whether a better fixed-count allocation is available at finite sample sizes, before training another learner.

Two experiments:
1. **Original model:** exact original generator and proposal, full batches of 84 observations, budget 4,032, all integer counts 4–120.
2. **Controlled heterogeneity:** hold observation noise and proposal width fixed while varying mixture separation or proposal-mean mismatch; include a fixed-scale control and a noise-only variant. Test whether the finite-sample oracle has headroom beyond inexpensive rules, not whether a learned model succeeds.

The target is $R_i(K)=d^{-1}\mathbb E\|\widehat\mu_{i,K}-\mu_i\|_2^2$. The true posterior mean is analytic in every testbed. The estimated oracle solves $\min_{\mathbf K}\sum_i\widehat R_i(K_i)$ under the integer budget. It sees privileged reference means and simulation banks. It is **not a deployable allocator or a guaranteed upper bound**. The true optimizer would be optimal among fixed-count policies with the same proposals. Evaluation uses independent random streams, and the dataset—not the draw—is the statistical unit.

**Run instructions:** open in Colab, run all cells in `smoke` mode first. Then set `MODE = "full"` and rerun all. `pilot` is exploratory. CPU is sufficient; no GPU required. Full mode is intentionally substantial; enable Drive persistence if desired. Do not use smoke/pilot outputs as confirmatory results. No external data or downloads needed.

In [ ]:
MODE = "full"       # "smoke", "pilot", or "full"
SAVE_TO_DRIVE = True # True: mount Google Drive for checkpoint/resume
# Full settings: 25 fresh complete batches per dimension per regime;
# two independent 1024-repetition optimization banks; 2048 evaluation repetitions.

In [ ]:
import os, json, time, math, heapq, hashlib, shutil, platform, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy.special import logsumexp
from scipy import stats
import matplotlib.pyplot as plt
import numba
from numba import njit
from IPython.display import display

PRESETS = {
    'smoke': dict(n=12, avg=48, kmin=4, kmax=120, dev_batches=2, test_batches=3, dev_reps=32, opt_reps=48, eval_reps=64, boot=300),
    'pilot': dict(n=84, avg=48, kmin=4, kmax=120, dev_batches=5, test_batches=8, dev_reps=256, opt_reps=512, eval_reps=1024, boot=3000),
    'full':  dict(n=84, avg=48, kmin=4, kmax=120, dev_batches=8, test_batches=25, dev_reps=512, opt_reps=1024, eval_reps=2048, boot=10000),
}
assert MODE in PRESETS
C = PRESETS[MODE].copy()
N, AVG, KMIN, KMAX = [C[k] for k in ('n','avg','kmin','kmax')]
BUDGET = N*AVG
KS = np.arange(KMIN,KMAX+1)
DIMS = [4,8]
REGIMES = ['original','fixed_scale','noise_only','separation_only','mismatch_only','combined_fixed_scale']
DEV_SEEDS = list(range(81001,81001+C['dev_batches']))
TEST_SEEDS = list(range(91001,91001+C['test_batches']))  # disjoint from old 1001/2001 cohorts
MASTER = 694201
GATE_PCT = 2.0
CHEAP = ['uniform','context_noise','proposal_width','separation','mismatch','ambiguity','scale_geometry']
# Revision identifier prevents mixing checkpoints from incompatible designs.
CONFIG = dict(version='stage0_v1', mode=MODE, **C, dimensions=DIMS, regimes=REGIMES, dev_seeds=DEV_SEEDS, test_seeds=TEST_SEEDS, master=MASTER, gate_pct=GATE_PCT, cheap=CHEAP)
TAG = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/importance_sampling_stage0')
else:
    ROOT = Path('/content/importance_sampling_stage0') if Path('/content').exists() else Path.cwd()/'importance_sampling_stage0'
OUT = ROOT/f'{MODE}_{TAG}'
OUT.mkdir(parents=True, exist_ok=True)
(OUT/'config.json').write_text(json.dumps(CONFIG,indent=2))
(OUT/'software.json').write_text(json.dumps(dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__,numba=numba.__version__),indent=2))
print('Mode:',MODE,'| Budget:',BUDGET,'| Output:',OUT)

## 1. Testbeds and analytical posterior
The original model is $z\sim\sum_{j=1}^2\pi_jN(m_j,I_d)$, $x\mid z,c\sim N(z,\sigma_c^2 I_d)$, with means $\pm2.25e_1$, weights $(0.55,0.45)$, noise $(0.45,1,2.1)$, and proposal $q=N(0.88x/(1+\sigma_c^2),uI_d)$, $u=\sigma_c^2a_c^2/(1+\sigma_c^2)$, $a_c=(1,1.1,1.25)$.

The original random generator is preserved. For the variants, all regimes share underlying random draws for a given seed/dimension (paired controls). The observed context includes separation and proposal offset magnitude where applicable; these are available to cheap rules. The prior is conditional on the known context, not an unobserved variable. The offsets vary only the proposal, never the target. All variants use the same two-component conditional posterior formula.

| Regime | Noise | Component half-separation | Proposal width | Proposal mean offset |
|---|---|---|---|---|
| original | original three levels | 2.25 | original | original 0.88 shrinkage |
| fixed_scale | 1 | 2.25 | fixed | zero |
| noise_only | three levels | 2.25 | posterior within-component SD × 1.1 | zero |
| separation_only | 1 | 0.5, 2.25, 4 | fixed | zero |
| mismatch_only | 1 | 2.25 | fixed | 0, 0.5, 1.5 within-component SDs |
| combined_fixed_scale | 1 | varied | fixed | varied |

`noise_only` changes proposal width through noise; it is not a width-independent causal effect. In fixed-scale variants, noise/width are constant but posterior scale and geometry can still vary. These deliberately controlled testbeds are not real-world validation. New results must be labeled by regime.

In [ ]:
def rng_for(*keys):
    return np.random.default_rng(np.random.SeedSequence([MASTER,*map(int,keys)]))

def make_model(d,seed,regime,n=N):
    rng=np.random.default_rng(seed)
    c=rng.integers(0,3,n)
    modes=rng.choice(2,n,p=[.55,.45])
    latent_eps=rng.normal(size=(n,d)); obs_eps=rng.normal(size=(n,d))
    ctx=rng_for(11,d,seed)
    sep_idx=ctx.integers(0,3,n); mismatch_idx=ctx.integers(0,3,n)
    sigma=np.array([.45,1.,2.1])[c] if regime in ('original','noise_only') else np.ones(n)
    sep=np.array([.5,2.25,4.])[sep_idx] if regime in ('separation_only','combined_fixed_scale') else np.full(n,2.25)
    means=np.zeros((n,2,d)); means[:,0,0]=-sep; means[:,1,0]=sep
    x=latent_eps+means[np.arange(n),modes]+sigma[:,None]*obs_eps
    v=sigma**2/(1+sigma**2)
    a=np.array([1.,1.1,1.25])[c] if regime=='original' else np.full(n,1.1)
    u=v*a*a
    nominal=x/(1+sigma[:,None]**2)
    qmu=nominal*.88 if regime=='original' else nominal.copy()
    if regime in ('mismatch_only','combined_fixed_scale'):
        # deterministic direction perpendicular to mixture axis; offset is known to allocator
        qmu[:,1]+=np.array([0.,.5,1.5])[mismatch_idx]*np.sqrt(v)
    mismatch=np.linalg.norm(qmu-nominal,axis=1)/np.sqrt(v)
    # Exact conditional posterior; all likelihood normalizers common across components.
    lg=np.log([.55,.45])[None,:]-.5*np.sum((x[:,None,:]-means)**2,axis=2)/(1+sigma[:,None]**2)
    lg-=logsumexp(lg,axis=1,keepdims=True)
    gamma=np.exp(lg)
    pm=(x[:,None,:]+sigma[:,None,None]**2*means)/(1+sigma[:,None,None]**2)
    truth=np.sum(gamma[:,:,None]*pm,axis=1)
    post_scale=np.sqrt(d*v+np.sum(gamma*np.sum((pm-truth[:,None,:])**2,axis=2),axis=1))
    return dict(d=d,x=x,c=c,sigma=sigma,sep=sep,v=v,u=u,qmu=qmu,mismatch=mismatch,lg=lg,pm=pm,truth=truth,post_scale=post_scale)

def exact_tau2(m):
    # Closed form integral of p^2/q times squared distance to true posterior mean.
    v,u,d=m['v'],m['u'],m['d']; A=2/v-1/u
    assert np.all(A>0), 'Asymptotic variance does not exist for this proposal.'
    a=m['pm']-m['qmu'][:,None,:]; target=m['truth']-m['qmu']; terms=[]
    for j,k in itertools.product(range(2),repeat=2):
        h=(a[:,j]+a[:,k])/v[:,None]; center=h/A[:,None]
        c=(np.sum(a[:,j]**2+a[:,k]**2,axis=1)/v)-np.sum(h*h,axis=1)/A
        logc=-d*np.log(2*np.pi*v)+.5*d*np.log(2*np.pi*u)+.5*d*np.log(2*np.pi/A)-.5*c
        moment=d/A+np.sum((center-target)**2,axis=1)
        terms.append(m['lg'][:,j]+m['lg'][:,k]+logc+np.log(moment))
    t=np.exp(logsumexp(np.stack(terms,axis=1),axis=1))
    assert np.all(np.isfinite(t)&(t>0))
    return t

# Verify exact original generator against direct transcription, not just summary statistics.
def legacy_x(d,seed,n):
    r=np.random.default_rng(seed); c=r.integers(0,3,n); sig=np.array([.45,1.,2.1])[c]
    means=np.zeros((2,d)); means[:,0]=[-2.25,2.25]
    modes=r.choice(2,n,p=[.55,.45]); z=r.normal(size=(n,d))+means[modes]
    return z+r.normal(size=(n,d))*sig[:,None],c
for d in DIMS:
    m=make_model(d,123,'original'); xx,cc=legacy_x(d,123,N)
    assert np.array_equal(m['x'],xx) and np.array_equal(m['c'],cc)
    assert np.allclose(np.exp(m['lg']).sum(1),1)
    assert np.allclose(m['qmu'],.88*xx/(1+m['sigma'][:,None]**2))
    for reg in REGIMES[1:]:
        m=make_model(d,123,reg)
        if reg!='noise_only': assert np.ptp(m['sigma'])==0 and np.ptp(m['u'])==0
    m=make_model(d,123,'fixed_scale'); m['pm'][:]=0; m['qmu'][:]=0; m['truth'][:]=0; m['u']=m['v'].copy()
    assert np.allclose(exact_tau2(m),d*m['v'])  # q=p duplicate-component Gaussian
print('Generator, controlled-scale and exact-variance tests passed.')

## 2. Independent Monte Carlo banks
For each observation, draw independent proposal trajectories of length 120, recording SNIS error at every prefix. Prefixes deliberately share draws across counts; independent trajectories are the Monte Carlo repetitions. The Numba kernel uses a running log-weight maximum to avoid numerical underflow. Normalized posterior weights are valid in SNIS because the evidence factor cancels.

Development banks choose the cheap comparator. For each test batch, optimization bank A chooses the primary oracle, independent bank B chooses a stability replicate, and A+B gives a larger-budget sensitivity analysis. A separate evaluation bank measures every frozen allocation with common random numbers. The evaluation bank must never choose a baseline, tune a parameter, or alter counts. Simulated bank costs are reported separately from the matched online budget.

In [ ]:
@njit
def prefix_errors(normals, qmu, u, pm, v, lg, truth, kmin):
    reps,kmax,d=normals.shape
    errors=np.empty((reps,kmax-kmin+1),np.float64)
    for r in range(reps):
        running_max=-np.inf; den=0.; num=np.zeros(d)
        for k in range(kmax):
            z=qmu+np.sqrt(u)*normals[r,k]
            d0=0.; d1=0.; dq=0.
            for j in range(d):
                d0+=(z[j]-pm[0,j])**2; d1+=(z[j]-pm[1,j])**2; dq+=normals[r,k,j]**2
            a=lg[0]-.5*d0/v; b=lg[1]-.5*d1/v
            mm=max(a,b)
            lw=mm+np.log(np.exp(a-mm)+np.exp(b-mm))+.5*d*np.log(u/v)+.5*dq
            newmax=max(running_max,lw)
            old=np.exp(running_max-newmax); new=np.exp(lw-newmax)
            den=old*den+new; num=old*num+new*z; running_max=newmax
            if k+1>=kmin:
                err=0.
                for j in range(d): err+=(num[j]/den-truth[j])**2
                errors[r,k+1-kmin]=err/d
    return errors

def risk_bank(m,seed,regime,role,reps):
    # Role and observation-specific independent random streams; same regime index across banks is never reused.
    risks=np.empty((len(m['x']),len(KS))); se=np.empty_like(risks)
    for i in range(len(risks)):
        # Common underlying normals across regimes support paired regime contrasts.
        rng=rng_for(40,m['d'],seed,role,i)
        e=prefix_errors(rng.standard_normal((reps,KMAX,m['d'])),m['qmu'][i],m['u'][i],m['pm'][i],m['v'][i],m['lg'][i],m['truth'][i],KMIN)
        risks[i]=e.mean(0); se[i]=e.std(0,ddof=1)/np.sqrt(reps)
    assert np.isfinite(risks).all() and np.all(risks>=0)
    return risks,se

# Kernel validation against vectorized direct SNIS, using the same normals.
m=make_model(4,42,'original'); i=0
z0=rng_for(99).standard_normal((16,KMAX,4)); z=m['qmu'][i]+np.sqrt(m['u'][i])*z0
lp=logsumexp(m['lg'][i][None,None,:]-.5*np.sum((z[:,:,None,:]-m['pm'][i][None,None,:,:])**2,axis=-1)/m['v'][i],axis=-1)
lw=lp+.5*4*np.log(m['u'][i]/m['v'][i])+.5*np.sum(z0*z0,axis=-1)
err=prefix_errors(z0,m['qmu'][i],m['u'][i],m['pm'][i],m['v'][i],m['lg'][i],m['truth'][i],KMIN)
for k in [KMIN,48,KMAX]:
    w=np.exp(lw[:,:k]-logsumexp(lw[:,:k],axis=1,keepdims=True)); est=np.sum(w[:,:,None]*z[:,:k],axis=1)
    assert np.allclose(err[:,k-KMIN],np.mean((est-m['truth'][i])**2,axis=1),rtol=1e-10,atol=1e-12)
print('SNIS prefix kernel verified against direct evaluation.')

## 3. Allocation policies and exact dynamic program
The inexpensive policies use the original capped proportional-extras heuristic (reserve $K_{\min}$ then distribute by score). Uniform is always included. Separation, known proposal offset, and an observation-derived ambiguity proxy are additional controls in the new regimes; their fixed formulae are stated in code, not tuned on test data. This makes the comparison harder than using only noise/width. The analytic posterior-scale and asymptotic variance policies use privileged information; report them separately. Exact asymptotic allocation uses the diminishing marginal gains $\tau_i^2/[K(K+1)]$.

For arbitrary finite-sample risk curves, use the exact recurrence $D_j(b)=\min_k[D_{j-1}(b-k)+\widehat R_j(k)]$. This does not assume monotonicity or convexity. We do not smooth the primary oracle curves; independent replication diagnoses noisy optimization. DP work is timed separately; no claim of negligible runtime is made.

In [ ]:
def capped_allocate(scores,total=BUDGET,lo=KMIN,hi=KMAX):
    scores=np.maximum(np.asarray(scores,float),1e-6); n=len(scores); extra=total-n*lo
    assert 0<=extra<=n*(hi-lo)
    if extra==0: return np.full(n,lo,int)
    left,right=0.,max(1.,extra/scores.min())
    for _ in range(60):
        mid=(left+right)/2
        if np.minimum(hi-lo,mid*scores).sum()<extra: left=mid
        else: right=mid
    e=np.minimum(hi-lo,right*scores); k=np.floor(e).astype(int)+lo
    rem=total-k.sum()
    if rem>0:
        order=np.argsort(-np.where(k<hi,e-np.floor(e),-np.inf),kind='stable'); k[order[:rem]]+=1
    elif rem<0:
        order=np.argsort(np.where(k>lo,e-np.floor(e),np.inf),kind='stable'); k[order[:(-rem)]]-=1
    assert k.sum()==total and k.min()>=lo and k.max()<=hi
    return k

@njit
def dp_allocate(risk,total,lo,hi):
    n=risk.shape[0]; extra=total-n*lo; width=hi-lo
    prev=np.full(extra+1,np.inf); prev[0]=0.
    choice=np.full((n,extra+1),-1,np.int16)
    for i in range(n):
        cur=np.full(extra+1,np.inf)
        bmin=max(0,extra-(n-i-1)*width); bmax=min(extra,(i+1)*width)
        for b in range(bmin,bmax+1):
            kmin=max(0,b-i*width); kmax=min(width,b)
            best=np.inf; arg=-1
            for k in range(kmin,kmax+1):
                val=prev[b-k]+risk[i,k]
                if val<best: best=val; arg=k
            cur[b]=best; choice[i,b]=arg
        prev=cur
    out=np.empty(n,np.int64); b=extra
    for i in range(n-1,-1,-1):
        k=choice[i,b]
        if k<0: raise ValueError('Infeasible DP')
        out[i]=k+lo; b-=k
    return out,prev[extra]

def risk_at(r,k,lo=KMIN): return r[np.arange(len(k)),k-lo]

def tau_allocate(tau2):
    k=np.full(len(tau2),KMIN,int)
    h=[(-float(t)/(KMIN*(KMIN+1)),i) for i,t in enumerate(tau2)]; heapq.heapify(h)
    for _ in range(BUDGET-k.sum()):
        _,i=heapq.heappop(h); k[i]+=1
        if k[i]<KMAX: heapq.heappush(h,(-float(tau2[i])/(k[i]*(k[i]+1)),i))
    return k

def policies(m):
    sigma,sep,u,mis=m['sigma'],m['sep'],m['u'],m['mismatch']
    # Simple fixed, non-learned proxies. Ambiguity uses observed coordinate, not posterior responsibilities.
    ambiguity=np.exp(-np.abs(m['x'][:,0])/np.sqrt(1+sigma*sigma))
    scores=dict(uniform=np.ones(len(sigma)),context_noise=sigma,proposal_width=np.sqrt(u),
        separation=np.sqrt(1+sep**2),mismatch=np.sqrt(1+mis**2),ambiguity=1+ambiguity,
        scale_geometry=np.sqrt(u)*(1+sep*ambiguity+mis))
    out={k:capped_allocate(v) for k,v in scores.items()}
    out['posterior_scale_oracle']=capped_allocate(m['post_scale'])
    tau2=exact_tau2(m); out['asymptotic_oracle']=tau_allocate(tau2)
    return out,tau2

# Exhaustive check on a nonmonotone tiny example and asymptotic greedy comparison.
r=np.array([[4.,2.,3.],[3.,4.,1.],[2.,1.,.5]])
k,val=dp_allocate(r,7,1,3)
ex=min(sum(r[i,a-1] for i,a in enumerate(ks)) for ks in itertools.product(range(1,4),repeat=3) if sum(ks)==7)
assert k.sum()==7 and np.isclose(val,ex)
m=make_model(4,123,'original'); pol,tau=policies(m)
k,val=dp_allocate(tau[:,None]/KS[None,:],BUDGET,KMIN,KMAX)
assert np.isclose(val,np.sum(tau/pol['asymptotic_oracle']))
print('DP verified by brute force and analytic greedy optimum.')

## 4. Freeze inexpensive comparator on development data
The best cheap rule is the one minimizing mean development-bank MSE, separately for each dimension/regime. Ties follow the fixed list order. Final test sets are not used in the selection. Posterior-scale and asymptotic oracles are diagnostic comparators, not eligible for selection. The original learned allocator is not retrained in Stage 0; its historical results are context only, not paired comparisons on these new datasets. The experiment tests headroom before learning.

In [ ]:
def atomic_csv(df,path):
    tmp=path.with_suffix('.tmp'); df.to_csv(tmp,index=False); tmp.replace(path)

def atomic_npz(path,**kwargs):
    tmp=path.with_suffix('.tmp.npz'); np.savez_compressed(tmp,**kwargs); tmp.replace(path)

p=OUT/'development.csv'
if p.exists():
    dev=pd.read_csv(p)
else:
    rows=[]
    for d in DIMS:
        for reg in REGIMES:
            for seed in DEV_SEEDS:
                m=make_model(d,seed,reg); r,_=risk_bank(m,seed,reg,1,C['dev_reps']); ps,_=policies(m)
                for name in CHEAP: rows.append(dict(dimension=d,regime=reg,seed=seed,method=name,mse=risk_at(r,ps[name]).mean()))
            print('Development:',d,reg,flush=True)
    dev=pd.DataFrame(rows); atomic_csv(dev,p)
assert len(dev)==len(DIMS)*len(REGIMES)*len(DEV_SEEDS)*len(CHEAP)
means=dev.groupby(['dimension','regime','method']).mse.mean().unstack().reindex(columns=CHEAP)
selected=means.idxmin(axis=1).to_dict()
selection=pd.DataFrame([dict(dimension=d,regime=r,selected_baseline=selected[d,r]) for d in DIMS for r in REGIMES])
atomic_csv(selection,OUT/'frozen_baselines.csv'); display(selection)

## 5. Main experiment: evaluate frozen allocations on independent banks
Every test batch is complete. The two optimization banks and evaluation bank have independent streams. The primary method uses bank A only. Bank B and pooled A+B are sensitivity analyses, not alternative primary outcomes. We record training-bank optimism, count instability, risk slopes, marginal-risk heterogeneity, exact budget checks and wall-clock times. Each batch is checkpointed in an NPZ file (contains risk curves, counts, errors, covariates and costs); safe to rerun with unchanged configuration. Never use evaluation curves to choose allocations.

In [ ]:
METHODS=CHEAP+['posterior_scale_oracle','asymptotic_oracle','finite_oracle_A','finite_oracle_B','finite_oracle_AB']

def run_batch(d,reg,seed):
    path=OUT/f'd{d}_{reg}_{seed}.npz'
    if path.exists(): return path
    m=make_model(d,seed,reg); t0=time.perf_counter()
    ra,sa=risk_bank(m,seed,reg,2,C['opt_reps']); rb,sb=risk_bank(m,seed,reg,3,C['opt_reps'])
    bank_time=time.perf_counter()-t0
    pol,tau=policies(m); t0=time.perf_counter()
    pol['finite_oracle_A'],_=dp_allocate(ra,BUDGET,KMIN,KMAX)
    dp_A_time=time.perf_counter()-t0; t0=time.perf_counter()
    pol['finite_oracle_B'],_=dp_allocate(rb,BUDGET,KMIN,KMAX)
    pol['finite_oracle_AB'],_=dp_allocate((ra+rb)/2,BUDGET,KMIN,KMAX)
    dp_other_time=time.perf_counter()-t0
    counts=np.stack([pol[name] for name in METHODS]); assert np.all(counts.sum(1)==BUDGET) and counts.min()>=KMIN and counts.max()<=KMAX
    t0=time.perf_counter(); re,se=risk_bank(m,seed,reg,4,C['eval_reps']); eval_time=time.perf_counter()-t0
    atomic_npz(path,methods=np.array(METHODS),counts=counts,ra=ra,rb=rb,re=re,sa=sa,sb=sb,se=se,tau2=tau,sigma=m['sigma'],sep=m['sep'],width=np.sqrt(m['u']),mismatch=m['mismatch'],post_scale=m['post_scale'],bank_time=bank_time,dp_A_time=dp_A_time,dp_other_time=dp_other_time,eval_time=eval_time)
    return path

paths=[]
for d in DIMS:
    for reg in REGIMES:
        t0=time.perf_counter()
        for j,seed in enumerate(TEST_SEEDS):
            paths.append(run_batch(d,reg,seed))
            if (j+1)%5==0 or j+1==len(TEST_SEEDS): print(f'{d}D | {reg} | {j+1}/{len(TEST_SEEDS)} | {time.perf_counter()-t0:.1f}s this regime',flush=True)
print('Completed batches:',len(paths))

## 6. Dataset-level results and oracle stability
All methods are evaluated under the same hypothetical online draw budget. Evaluation-bank trajectories estimate expected risk, not deployed wall-clock inference. We aggregate over observations within each independent dataset before statistical inference. Report relative improvement as $100(1-\overline{MSE}_{method}/\overline{MSE}_{base})$; bootstrap paired datasets for its interval, and use a paired t interval on absolute MSE difference. A/B count disagreement and pooled-bank sensitivity indicate optimization noise. For marginal-risk diagnostics, use a 16-draw difference (40 to 56), not noisy adjacent differences. Local power-law exponents are descriptive, not assumed in allocation.

In [ ]:
rows=[]; diags=[]; observations=[]
for d in DIMS:
    for reg in REGIMES:
        for seed in TEST_SEEDS:
            with np.load(OUT/f'd{d}_{reg}_{seed}.npz',allow_pickle=False) as b:
                counts=dict(zip(b['methods'].tolist(),b['counts'])); ra,rb,re=b['ra'],b['rb'],b['re']; base=selected[d,reg]
                for name,k in counts.items():
                    rows.append(dict(dimension=d,regime=reg,seed=seed,method=name,mse=risk_at(re,k).mean(),opt_A_mse=risk_at(ra,k).mean(),opt_B_mse=risk_at(rb,k).mean(),budget=k.sum(),k_min=k.min(),k_max=k.max(),selected_baseline=base))
                ka,kb=counts['finite_oracle_A'],counts['finite_oracle_B']; k0=counts[base]; kab=counts['finite_oracle_AB']
                asym=b['tau2']/(d*48)
                r48=re[:,48-KMIN]; slope=-np.log(np.maximum(re[:,120-KMIN],1e-300)/np.maximum(r48,1e-300))/np.log(120/48)
                marginal=(re[:,40-KMIN]-re[:,56-KMIN])/16
                diags.append(dict(dimension=d,regime=reg,seed=seed,selected_baseline=base,
                    count_l1_AB=np.abs(ka-kb).sum(),count_mean_abs_AB=np.abs(ka-kb).mean(),count_equal_AB=(ka==kb).mean(),
                    train_gain_A=100*(1-risk_at(ra,ka).mean()/risk_at(ra,k0).mean()),
                    eval_gain_A=100*(1-risk_at(re,ka).mean()/risk_at(re,k0).mean()),
                    eval_gain_B=100*(1-risk_at(re,kb).mean()/risk_at(re,k0).mean()),
                    eval_gain_AB=100*(1-risk_at(re,kab).mean()/risk_at(re,k0).mean()),
                    r48_cv=r48.std()/r48.mean(),marginal_sd=marginal.std(),slope_mean=slope.mean(),slope_sd=slope.std(),
                    risk_asym_ratio=r48.mean()/asym.mean(),tau_risk_spearman=stats.spearmanr(b['tau2'],r48).statistic,
                    opt_density_evals=2*N*C['opt_reps']*KMAX,eval_density_evals=N*C['eval_reps']*KMAX,
                    bank_seconds=float(b['bank_time']),dp_A_seconds=float(b['dp_A_time']),dp_other_seconds=float(b['dp_other_time']),eval_seconds=float(b['eval_time'])))
                for i in range(N):
                    observations.append(dict(dimension=d,regime=reg,seed=seed,observation=i,sigma=b['sigma'][i],separation=b['sep'][i],proposal_width=b['width'][i],proposal_mismatch=b['mismatch'][i],posterior_scale=b['post_scale'][i],tau2=b['tau2'][i],risk48=r48[i],risk48_mcse=b['se'][i,48-KMIN],local_beta=slope[i],marginal_40_56=marginal[i],K_baseline=k0[i],K_oracle_A=ka[i],K_oracle_B=kb[i],K_oracle_AB=kab[i]))
trials=pd.DataFrame(rows); diagnostics=pd.DataFrame(diags); obs=pd.DataFrame(observations)
for name,df in [('batch_results',trials),('oracle_diagnostics',diagnostics),('observation_diagnostics',obs)]: atomic_csv(df,OUT/f'{name}.csv')
assert (trials.budget==BUDGET).all()

In [ ]:
def paired_summary(base,method,seed):
    base=np.asarray(base); method=np.asarray(method); delta=base-method; n=len(base)
    gain=100*(1-method.mean()/base.mean())
    se=delta.std(ddof=1)/np.sqrt(n); crit=stats.t.ppf(.975,n-1)
    lo,hi=delta.mean()-crit*se,delta.mean()+crit*se
    # One-sided paired t p-value for positive improvement; report exploratory multiplicity adjustment below.
    p=stats.t.sf(delta.mean()/se,n-1) if se>0 else (0. if delta.mean()>0 else 1.)
    inds=rng_for(88,seed).integers(0,n,(C['boot'],n)); g=100*(1-method[inds].mean(1)/base[inds].mean(1))
    return dict(n_batches=n,baseline_mse=base.mean(),method_mse=method.mean(),gain_pct=gain,gain_ci_lo=np.quantile(g,.025),gain_ci_hi=np.quantile(g,.975),delta_mse=delta.mean(),delta_ci_lo=lo,delta_ci_hi=hi,p_one_sided=p,batches_better=int((delta>0).sum()))

summary=[]
for (d,reg),g in trials.groupby(['dimension','regime'],sort=False):
    w=g.pivot(index='seed',columns='method',values='mse').sort_index(); base=selected[d,reg]
    for j,name in enumerate(METHODS):
        summary.append(dict(dimension=d,regime=reg,method=name,baseline=base,**paired_summary(w[base],w[name],d*100+REGIMES.index(reg)*20+j),gain_vs_uniform_pct=100*(1-w[name].mean()/w['uniform'].mean())))
summary=pd.DataFrame(summary); atomic_csv(summary,OUT/'method_summary.csv')
primary=summary[summary.method=='finite_oracle_A'].copy().reset_index(drop=True)
# Holm across all prespecified dimension/regime cells, including exploratory controlled variants.
p=primary.p_one_sided.to_numpy(); order=np.argsort(p); adj=np.empty(len(p)); adj[order]=np.minimum(1,np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order])); primary['holm_p']=adj
primary['point_gate']=(primary.gain_pct>=GATE_PCT)&(primary.delta_ci_lo>0)&(primary.holm_p<.05)
primary['strong_gate']=primary.point_gate&(primary.gain_ci_lo>=GATE_PCT)
primary['decision']=np.where(primary.point_gate,'HEADROOM SUPPORTED','NOT ESTABLISHED') if MODE=='full' else 'EXPLORATORY ONLY'
atomic_csv(primary,OUT/'gate_summary.csv')
display(primary[['dimension','regime','baseline','gain_pct','gain_ci_lo','gain_ci_hi','batches_better','holm_p','point_gate','decision']].round(4))
print('Gate: >=2% point gain + positive paired 95% absolute difference interval + Holm p<0.05; stronger gate requires relative CI lower bound >=2%.')
if MODE!='full': print('Smoke/pilot mode cannot establish the confirmatory gate.')

## 7. Controlled heterogeneity analysis
Compare each controlled regime with the fixed-scale control using paired seeds. Changes in oracle headroom are estimated as differences in ratios of cohort means, not the difference between independent confidence intervals. Since the cheapest baseline can differ by regime, this contrast answers how attainable gain over each regime's selected cheap policy changes. It is not a pure causal estimate of a single latent geometric property. In addition, report calibration, risk and marginal heterogeneity, stability and runtime. Exploratory contrasts are not extra confirmatory claims.

In [ ]:
contrast=[]
for d in DIMS:
    def paired_arrays(reg):
        w=trials[(trials.dimension==d)&(trials.regime==reg)].pivot(index='seed',columns='method',values='mse').sort_index()
        return w[selected[d,reg]].to_numpy(),w.finite_oracle_A.to_numpy()
    b0,m0=paired_arrays('fixed_scale')
    for j,reg in enumerate(['noise_only','separation_only','mismatch_only','combined_fixed_scale']):
        b,m=paired_arrays(reg); n=len(b)
        idx=rng_for(96,d,j).integers(0,n,(C['boot'],n))
        point=100*(m0.mean()/b0.mean()-m.mean()/b.mean())
        dist=100*(m0[idx].mean(1)/b0[idx].mean(1)-m[idx].mean(1)/b[idx].mean(1))
        contrast.append(dict(dimension=d,regime=reg,reference='fixed_scale',headroom_change_pp=point,ci_lo=np.quantile(dist,.025),ci_hi=np.quantile(dist,.975)))
contrasts=pd.DataFrame(contrast); atomic_csv(contrasts,OUT/'heterogeneity_contrasts.csv'); display(contrasts.round(3))
cols=['count_mean_abs_AB','train_gain_A','eval_gain_A','eval_gain_B','eval_gain_AB','r48_cv','marginal_sd','slope_mean','slope_sd','risk_asym_ratio','tau_risk_spearman','dp_A_seconds','bank_seconds','eval_seconds']
stability=diagnostics.groupby(['dimension','regime'])[cols].mean().reset_index(); atomic_csv(stability,OUT/'stability_summary.csv'); display(stability.round(4))

## 8. Figures for the report
All plots are generated from the current run, never copied from older experiments. Confidence intervals are dataset-level paired bootstrap intervals. Illustrative diagnostics of risk curves are computed over complete held-out batches. The exact risk curves are stored in the per-batch NPZ checkpoints.

In [ ]:
FIG=OUT/'figures'; FIG.mkdir(exist_ok=True)
for d in DIMS:
    p=primary[primary.dimension==d].set_index('regime').loc[REGIMES]
    fig,ax=plt.subplots(figsize=(9,4.8)); y=np.arange(len(p)); x=p.gain_pct.to_numpy()
    ax.errorbar(x,y,xerr=[np.maximum(0,x-p.gain_ci_lo),np.maximum(0,p.gain_ci_hi-x)],fmt='o',capsize=4)
    ax.axvline(0,color='gray',ls='--'); ax.axvline(GATE_PCT,color='black',ls=':'); ax.set_yticks(y,p.index); ax.invert_yaxis(); ax.set_xlabel('Oracle MSE reduction vs frozen cheap baseline (%)'); ax.set_title(f'{d}D — {MODE} | independent evaluation'); fig.tight_layout(); fig.savefig(FIG/f'headroom_{d}D.png',dpi=180); plt.show(); plt.close(fig)
    g=summary[(summary.dimension==d)&(summary.regime=='original')].sort_values('gain_vs_uniform_pct')
    fig,ax=plt.subplots(figsize=(8,5.4)); ax.barh(g.method,g.gain_vs_uniform_pct); ax.axvline(0,color='gray'); ax.set_xlabel('MSE reduction vs uniform (%)'); ax.set_title(f'Original model, {d}D — {MODE}'); fig.tight_layout(); fig.savefig(FIG/f'original_methods_{d}D.png',dpi=180); plt.show(); plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,4.7))
    for reg in REGIMES:
        curves=[]
        for seed in TEST_SEEDS:
            with np.load(OUT/f'd{d}_{reg}_{seed}.npz') as b: curves.append(b['re'].mean(0))
        mean=np.mean(curves,axis=0); ax.plot(KS,mean/mean[48-KMIN],label=reg)
    ax.plot(KS,48/KS,'k--',label='1/K reference'); ax.set(xlabel='Samples K',ylabel='Mean risk / mean risk at K=48',yscale='log',title=f'Finite-sample risk shape, {d}D — {MODE}'); ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(FIG/f'risk_shapes_{d}D.png',dpi=180); plt.show(); plt.close(fig)
    g=stability[stability.dimension==d].set_index('regime').loc[REGIMES]
    fig,ax=plt.subplots(figsize=(9,4.5)); xx=np.arange(len(g));
    for j,name in enumerate(['eval_gain_A','eval_gain_B','eval_gain_AB']): ax.bar(xx+(j-1)*.25,g[name],width=.25,label=name)
    ax.axhline(0,color='gray'); ax.set_xticks(xx,g.index,rotation=25,ha='right'); ax.set_ylabel('Mean per-batch gain (%)'); ax.set_title(f'Independent oracle bank stability, {d}D — {MODE}'); ax.legend(); fig.tight_layout(); fig.savefig(FIG/f'bank_stability_{d}D.png',dpi=180); plt.show(); plt.close(fig)

## 9. Interpret the outcome and export
- Full-mode positive gate: evidence of available headroom for fixed-count allocation in that specified testbed, not evidence that a learned predictor captures it.
- Positive A+B but unstable A/B: increase independent optimization replication in a **new** prespecified sensitivity run. Do not silently replace the primary result.
- No positive gate: report no demonstrated headroom, not proof of absence; examine precision, cheap baselines, and controlled settings. No post hoc changes to make the test pass.
- Strong gain only in scale-matched regimes: conditional evidence that heterogeneity beyond noise/width can matter; not proof of real-world usefulness.
- High oracle cost is offline research cost, not deployment savings. DP timing is measured, but actual online latency and learned-predictor break-even require subsequent experiments.

The final archive contains CSVs, plots, configuration, software versions and all per-batch risk arrays. In full mode it may be sizable. Download it after the run and share the CSVs for interpretation. Do not report the historical learned allocator as a new matched comparison. No new training is performed here.

In [ ]:
if MODE=='full':
    if primary.point_gate.any():
        print('Stage 0: supported in the following prespecified cells:')
        display(primary.loc[primary.point_gate,['dimension','regime','gain_pct','gain_ci_lo','gain_ci_hi','holm_p']])
    else: print('Stage 0: no prespecified cell passed; report this outcome and uncertainty.')
else: print('Exploratory run only; no confirmatory decision.')
print('Development simulation draws:',len(DIMS)*len(REGIMES)*len(DEV_SEEDS)*N*C['dev_reps']*KMAX)
print('Test optimization draws:',int(diagnostics.opt_density_evals.sum()),'| Test evaluation draws:',int(diagnostics.eval_density_evals.sum()))
print('These are offline simulation counts, not online budget or measured deployment speedup.')
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Results archive:',archive)
# In Colab, uncomment to download if desired:
# from google.colab import files
# files.download(archive)